In [1]:
import uuid
import pandas as pd
import qnexus as qnx
from IPython.display import display

from funciones_nexus import (
    conectar_nexus,
    obtener_execution_jobs,
    construir_selector_jobs,
    resumir_job,
    descargar_resultados_job,
    construir_tabla_resultados,
    compilar_y_subir_hugr,
    ejecutar_local,
    enviar_job_selene,
    nuevo_run_id_local,
    guardar_ejecucion_csv,
)

PROJECT_NAME = "guppy-kernel-encoding"          # Nombre exacto del proyecto existente

# Interruptor maestro del cuaderno:
#   True  -> RUN NUEVO: se ejecuta una simulacion nueva (local o remota).
#            NO se consultan jobs existentes (las celdas de listado/seleccion no aplican).
#   False -> CONSULTA: se lista y se carga un job ya terminado especifico.
#            NO se ejecuta nada nuevo (las celdas de compilacion/ejecucion no aplican).
ALLOW_NEW_EXECUTION = True

# Destino del run nuevo (solo se usa cuando ALLOW_NEW_EXECUTION = True):
#   "local"        -> simulador local (no toca Nexus)
#   "nexus_selene" -> job remoto en Nexus/Selene
EXECUTION_TARGET = "local"

n_shots = 100

RESULT_SOURCE = None
local_result = None
local_counts = None
local_run_id = None
sim_job_ref = None
sim_result = None
sim_counts = None
sim_result_ids = None
suffix = uuid.uuid4().hex[:8]

project = conectar_nexus(PROJECT_NAME)

print("Conexion con Nexus comprobada.")
print(f"Proyecto: {PROJECT_NAME}")
print(f"Project ID: {project.id}")
print("Modo:", "RUN NUEVO" if ALLOW_NEW_EXECUTION else "CONSULTA de job existente")
if ALLOW_NEW_EXECUTION:
    print("Destino:", EXECUTION_TARGET)


Already logged in. Tokens are valid.
Conexion con Nexus comprobada.
Proyecto: guppy-kernel-encoding
Project ID: a9629379-d971-4ea6-af61-5e4b69b20852
Modo: RUN NUEVO
Destino: local


In [2]:
# Listar jobs existentes solo aplica en modo CONSULTA (ALLOW_NEW_EXECUTION = False).
# En modo RUN NUEVO no se consulta nada: se hara una ejecucion nueva.

execution_job_refs = []
job_selector = None

if ALLOW_NEW_EXECUTION:
    print("No aplica: ALLOW_NEW_EXECUTION = True (run nuevo completo); no se consultan jobs existentes.")

else:
    execution_job_refs = obtener_execution_jobs(project)
    print(f"Jobs disponibles en {PROJECT_NAME}: {len(execution_job_refs)}")
    job_selector = construir_selector_jobs(execution_job_refs)
    display(job_selector)
    print("Selecciona un job y luego ejecuta la siguiente celda.")


No aplica: ALLOW_NEW_EXECUTION = True (run nuevo completo); no se consultan jobs existentes.


In [3]:
# Cargar los resultados de un job existente solo aplica en modo CONSULTA.
# En modo RUN NUEVO no hay resultados previos que cargar.

if ALLOW_NEW_EXECUTION:
    print("No aplica: ALLOW_NEW_EXECUTION = True (run nuevo completo); no se cargan resultados previos.")

else:
    SELECTED_JOB_INDEX = job_selector.value

    if SELECTED_JOB_INDEX is None:
        raise ValueError("No hay jobs disponibles para seleccionar.")
    if not 0 <= SELECTED_JOB_INDEX < len(execution_job_refs):
        raise IndexError(f"Indice fuera de rango: {SELECTED_JOB_INDEX}")

    selected_job_ref = execution_job_refs[SELECTED_JOB_INDEX]
    selected_job_name, selected_job_status = resumir_job(selected_job_ref)

    print(f"Indice seleccionado: {SELECTED_JOB_INDEX}")
    print(f"Job: {selected_job_name}")
    print(f"Job ID: {selected_job_ref.id}")
    print(f"Estado: {selected_job_status}")

    # Descarga y tabula los resultados (valida que el job este COMPLETED).
    selected_result_refs, selected_downloaded_results, selected_counts, selected_result_ids = (
        descargar_resultados_job(selected_job_ref)
    )

    selected_results_df = construir_tabla_resultados(selected_counts, selected_result_ids)
    display(selected_results_df)

    sim_result = selected_downloaded_results[0]
    sim_counts = selected_counts[0] if len(selected_counts) == 1 else selected_counts
    RESULT_SOURCE = "selected_nexus_job"


No aplica: ALLOW_NEW_EXECUTION = True (run nuevo completo); no se cargan resultados previos.


In [4]:
from guppylang import guppy
from guppylang.std.builtins import result
from guppylang.std.quantum import cx, h, measure, qubit


@guppy
def encode_logical_plus() -> None:
    data, parity = qubit(), qubit()
    h(data)
    cx(data, parity)
    result("logical[0]", measure(data))
    result("logical[1]", measure(parity))


encode_logical_plus.check()


In [5]:
# Compila y sube HUGR solo para un RUN NUEVO remoto (nexus_selene).
# En modo CONSULTA no aplica; en modo local la compilacion ocurre dentro de la simulacion.

hugr_binary = None
ref_hugr = None

if not ALLOW_NEW_EXECUTION:
    print("No aplica: modo CONSULTA (ALLOW_NEW_EXECUTION = False); no se compila ni sube HUGR.")

elif EXECUTION_TARGET == "nexus_selene":
    hugr_binary, ref_hugr = compilar_y_subir_hugr(
        encode_logical_plus, f"encoding-logical-plus-{suffix}"
    )
    print("HUGR compilado y subido a Nexus:", ref_hugr)

elif EXECUTION_TARGET == "local":
    print("Modo local: la compilacion ocurre dentro de la simulacion; no se sube HUGR a Nexus.")

else:
    raise ValueError('EXECUTION_TARGET debe ser "local" o "nexus_selene"')


Modo local: la compilacion ocurre dentro de la simulacion; no se sube HUGR a Nexus.


In [6]:
# Ejecuta un RUN NUEVO segun el destino:
#   - modo CONSULTA (ALLOW_NEW_EXECUTION = False): no ejecuta nada
#   - local: corre el simulador local (compila internamente)
#   - nexus_selene: envia el job remoto y termina inmediatamente

if not ALLOW_NEW_EXECUTION:
    print("Modo CONSULTA: no se ejecuta simulacion nueva; se usan los resultados cargados arriba.")

elif EXECUTION_TARGET == "local":                                # Camino rapido: sin subir a queue
    local_result, local_counts = ejecutar_local(
        encode_logical_plus, n_qubits=2, n_shots=n_shots, seed=42
    )
    RESULT_SOURCE = "local"
    local_run_id = nuevo_run_id_local()                          # Id unico de esta ejecucion local
    print("Simulacion local finalizada correctamente.")
    local_counts                                                  # Muestra los conteos agrupados

elif EXECUTION_TARGET == "nexus_selene":                         # Camino remoto: Selene en Nexus
    if ref_hugr is None:
        raise RuntimeError("ref_hugr no existe; ejecuta primero la celda de compilacion/upload.")

    sim_job_ref = enviar_job_selene(
        ref_hugr, n_qubits=2, n_shots=n_shots, nombre=f"encoding-selene-sim-{suffix}"
    )
    submit_status = qnx.jobs.status(sim_job_ref)                  # Consulta estado inmediatamente despues del submit

    print("Job enviado a Selene/Nexus correctamente.")
    print("sim_job_ref:", sim_job_ref)
    print("Estado inicial:", submit_status.status)
    print("Mensaje:", submit_status.message)
    print("La celda termina aqui; consulta el avance en la siguiente celda.")

else:
    raise ValueError('EXECUTION_TARGET debe ser "local" o "nexus_selene"')


Simulacion local finalizada correctamente.


In [7]:
# Consulta el estado de un job remoto nuevo sin bloquear el notebook.
# Solo aplica si en la celda anterior se envio un job remoto (nexus_selene).
# Reejecuta esta celda manualmente cada vez que quieras revisar avance.

if sim_job_ref is None:                                           # No hay job remoto nuevo en esta sesion
    print("No hay sim_job_ref que consultar. Estas en modo local, en modo CONSULTA, o aun no enviaste un job remoto.")

else:
    status = qnx.jobs.status(sim_job_ref)                         # Consulta estado actual del job en Nexus

    print("Estado:", status.status)                               # Imprime estado resumido
    print("Mensaje:", status.message)                             # Imprime mensaje descriptivo

    queue_position = getattr(status, "queue_position", None)       # Lee posicion en cola si existe
    if queue_position is not None:
        print("Posicion en cola:", queue_position)

    if "COMPLETED" in str(status.status):                          # Si termino, descarga resultados
        sim_result_refs, sim_downloaded_results, sim_counts_list, sim_result_ids = (
            descargar_resultados_job(sim_job_ref)
        )
        sim_result = sim_downloaded_results[0]                     # Compatibilidad: primer resultado
        sim_counts = sim_counts_list[0] if len(sim_counts_list) == 1 else sim_counts_list
        RESULT_SOURCE = "new_nexus_job"
        print("Job finalizado. Resultado descargado en sim_result; conteos en sim_counts.")
        sim_counts

    elif "ERROR" in str(status.status) or "CANCELLED" in str(status.status):
        raise RuntimeError(f"El job termino sin exito: {status}")

    else:
        print("Job aun no finalizado. Vuelve a ejecutar esta celda mas tarde.")


No hay sim_job_ref que consultar. Estas en modo local, en modo CONSULTA, o aun no enviaste un job remoto.


In [8]:
# Utilidad final: guarda los resultados de la ejecucion actual en un CSV con formato unico.
# Cubre los tres casos: simulador local, job de Nexus reutilizado (seleccionado hoy) y
# job de Nexus recien terminado con el cuaderno abierto. Todos comparten el mismo formato
# de tabla (el de Nexus) para poder analizarlos juntos. Los CSV se guardan en data/runs/.
# Para jobs de Nexus el archivo se nombra por el id del job, asi que volver a guardar el
# mismo job REEMPLAZA su CSV en vez de duplicarlo.

if RESULT_SOURCE == "local":
    if local_counts is None:
        raise RuntimeError("No hay resultados locales que guardar (local_counts esta vacio).")
    ruta_run = guardar_ejecucion_csv(
        counts=local_counts,
        source="local",
        n_shots=n_shots,
        run_id=local_run_id,
    )

elif RESULT_SOURCE == "selected_nexus_job":
    ruta_run = guardar_ejecucion_csv(
        counts=selected_counts,
        source="selected_nexus_job",
        job_id=selected_job_ref.id,
        job_name=selected_job_name,
        result_ids=selected_result_ids,
        n_shots=n_shots,
    )

elif RESULT_SOURCE == "new_nexus_job":
    ruta_run = guardar_ejecucion_csv(
        counts=sim_counts,
        source="new_nexus_job",
        job_id=sim_job_ref.id,
        job_name=getattr(sim_job_ref.annotations, "name", None) or str(sim_job_ref.id),
        result_ids=sim_result_ids,
        n_shots=n_shots,
    )

else:
    raise RuntimeError(
        "No hay una ejecucion cargada para guardar. Corre una simulacion local, "
        "selecciona un job de Nexus, o espera a que termine un job nuevo."
    )

print(f"Ejecucion guardada en: {ruta_run}")
pd.read_csv(ruta_run)

Ejecucion guardada en: data/runs\run_local-73c717914295.csv


,run_id,source,job_id,job_name,result_index,result_id,outcome,count,shots,proportion
0,local-73c717914295,local,NaN,NaN,0,NaN,logical[0]=0 | logical[1]=0,56,100,0.56
1,local-73c717914295,local,NaN,NaN,0,NaN,logical[0]=1 | logical[1]=1,44,100,0.44
